In [2]:
# ================================================
# Cookie Cats A/B Test Analysis
# ================================================
# Business Question:
# Should we move the first gate from level 30 to level 40?
# Does it significantly change Day-1 and Day-7 player retention?
#
# Dataset: ~90,000 players
# Control:   gate_30 (old design)
# Treatment: gate_40 (new design)
# ================================================

In [3]:
#Cell 2 — Load all libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [4]:
#Cell 3 — Load the data:
df = pd.read_csv('../data/cookie_cats.csv')

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Data loaded successfully!
Shape: (90189, 5)
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [5]:
# ================================================
# STEP 2: Exploratory Data Analysis
# ================================================

print("=== BASIC DATASET INFO ===")
print(f"Total players: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

=== BASIC DATASET INFO ===
Total players: 90,189
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']

Data types:
userid             int64
version           object
sum_gamerounds     int64
retention_1         bool
retention_7         bool
dtype: object

Missing values:
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64


In [6]:
#Cell 5 — Group sizes
print("=== GROUP SIZES ===")
group_counts = df['version'].value_counts()
print(group_counts)
print(f"\nPercentage split:")
print(df['version'].value_counts(normalize=True).mul(100).round(2))

=== GROUP SIZES ===
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

Percentage split:
version
gate_40    50.44
gate_30    49.56
Name: proportion, dtype: float64


In [7]:
#Cell 6 — Game rounds analysis
print("=== GAME ROUNDS ANALYSIS ===")
print(f"Average rounds played: {df['sum_gamerounds'].mean():.1f}")
print(f"Median rounds played: {df['sum_gamerounds'].median():.1f}")
print(f"Max rounds played: {df['sum_gamerounds'].max():,}")
print(f"Min rounds played: {df['sum_gamerounds'].min()}")
print(f"\nPlayers who played 0 rounds: {(df['sum_gamerounds'] == 0).sum():,}")
print(f"Players who played 1-10 rounds: {((df['sum_gamerounds'] >= 1) & (df['sum_gamerounds'] <= 10)).sum():,}")
print(f"Players who played 100+ rounds: {(df['sum_gamerounds'] >= 100).sum():,}")

=== GAME ROUNDS ANALYSIS ===
Average rounds played: 51.9
Median rounds played: 16.0
Max rounds played: 49,854
Min rounds played: 0

Players who played 0 rounds: 3,994
Players who played 1-10 rounds: 31,995
Players who played 100+ rounds: 12,516


In [9]:
#Cell 7 — Retention rates
print("=== RETENTION RATES BY GROUP ===")

retention = df.groupby('version')[['retention_1', 'retention_7']].mean().mul(100).round(2)
print(retention)

print(f"\n--- Plain English Summary ---")
print(f"Day-1 Retention:")
print(f"  gate_30: {retention.loc['gate_30', 'retention_1']}% of players came back the next day")
print(f"  gate_40: {retention.loc['gate_40', 'retention_1']}% of players came back the next day")

print(f"\nDay-7 Retention:")
print(f"  gate_30: {retention.loc['gate_30', 'retention_7']}% of players came back after 7 days")
print(f"  gate_40: {retention.loc['gate_40', 'retention_7']}% of players came back after 7 days")

=== RETENTION RATES BY GROUP ===
         retention_1  retention_7
version                          
gate_30        44.82        19.02
gate_40        44.23        18.20

--- Plain English Summary ---
Day-1 Retention:
  gate_30: 44.82% of players came back the next day
  gate_40: 44.23% of players came back the next day

Day-7 Retention:
  gate_30: 19.02% of players came back after 7 days
  gate_40: 18.2% of players came back after 7 days


In [10]:
# ================================================
# STEP 3: Experiment Validation
# ================================================

print("=== CHECK 1: SAMPLE RATIO MISMATCH (SRM) ===")

# Count players in each group
n_gate30 = len(df[df['version'] == 'gate_30'])
n_gate40 = len(df[df['version'] == 'gate_40'])
total = len(df)

print(f"gate_30 players: {n_gate30:,}")
print(f"gate_40 players: {n_gate40:,}")
print(f"Total players:   {total:,}")
print(f"Expected per group (50/50): {total//2:,}")
print(f"Difference from expected: {abs(n_gate30 - n_gate40):,} players")

# Calculate the ratio
ratio = n_gate40 / n_gate30
print(f"\nRatio (gate_40/gate_30): {ratio:.4f}")
print(f"Perfect ratio would be: 1.0000")
print(f"Difference from perfect: {abs(1 - ratio):.4f}")

if abs(1 - ratio) < 0.01:
    print("\n✅ No Sample Ratio Mismatch — groups are balanced")
else:
    print("\n❌ WARNING: Sample Ratio Mismatch detected!")

=== CHECK 1: SAMPLE RATIO MISMATCH (SRM) ===
gate_30 players: 44,700
gate_40 players: 45,489
Total players:   90,189
Expected per group (50/50): 45,094
Difference from expected: 789 players

Ratio (gate_40/gate_30): 1.0177
Perfect ratio would be: 1.0000
Difference from perfect: 0.0177

❌ WARNING: Sample Ratio Mismatch detected!


In [11]:
print("=== CHECK 2: ZERO ROUND PLAYERS ===")

# Find players who never played
zero_players = df[df['sum_gamerounds'] == 0]
zero_gate30 = len(zero_players[zero_players['version'] == 'gate_30'])
zero_gate40 = len(zero_players[zero_players['version'] == 'gate_40'])

print(f"Total players who played 0 rounds: {len(zero_players):,}")
print(f"  gate_30: {zero_gate30:,} ({zero_gate30/n_gate30*100:.2f}% of gate_30)")
print(f"  gate_40: {zero_gate40:,} ({zero_gate40/n_gate40*100:.2f}% of gate_40)")

print(f"\nRetention of 0-round players:")
print(zero_players[['retention_1', 'retention_7']].mean().mul(100).round(2))

print(f"\nShould we remove them?")
print(f"These players never saw the gate — including them adds noise to our analysis")

=== CHECK 2: ZERO ROUND PLAYERS ===
Total players who played 0 rounds: 3,994
  gate_30: 1,937 (4.33% of gate_30)
  gate_40: 2,057 (4.52% of gate_40)

Retention of 0-round players:
retention_1    2.18
retention_7    0.73
dtype: float64

Should we remove them?
These players never saw the gate — including them adds noise to our analysis


In [12]:
print("=== CHECK 3: OUTLIER CHECK ===")

print(f"Game rounds distribution:")
print(df['sum_gamerounds'].describe().round(2))

# Check extreme players
print(f"\nPlayers with 1000+ rounds: {(df['sum_gamerounds'] >= 1000).sum():,}")
print(f"Players with 2000+ rounds: {(df['sum_gamerounds'] >= 2000).sum():,}")
print(f"Players with 5000+ rounds: {(df['sum_gamerounds'] >= 5000).sum():,}")

# Are outliers evenly distributed between groups?
outliers_gate30 = len(df[(df['version'] == 'gate_30') & (df['sum_gamerounds'] >= 1000)])
outliers_gate40 = len(df[(df['version'] == 'gate_40') & (df['sum_gamerounds'] >= 1000)])

print(f"\n1000+ round players by group:")
print(f"  gate_30: {outliers_gate30:,}")
print(f"  gate_40: {outliers_gate40:,}")

=== CHECK 3: OUTLIER CHECK ===
Game rounds distribution:
count    90189.00
mean        51.87
std        195.05
min          0.00
25%          5.00
50%         16.00
75%         51.00
max      49854.00
Name: sum_gamerounds, dtype: float64

Players with 1000+ rounds: 118
Players with 2000+ rounds: 10
Players with 5000+ rounds: 1

1000+ round players by group:
  gate_30: 53
  gate_40: 65


In [13]:
# ================================================
# CREATING CLEANED DATASET
# ================================================

# Remove players who played 0 rounds
# Reason: they never reached the gate we're testing
# so their behavior tells us nothing about the gate's effect

df_clean = df[df['sum_gamerounds'] > 0].copy()

# .copy() creates a brand new independent copy of the filtered data
# Without .copy(), changes to df_clean might accidentally affect df
# Think of it like photocopying a document vs editing the original

print(f"Original dataset:  {len(df):,} players")
print(f"Cleaned dataset:   {len(df_clean):,} players")
print(f"Players removed:   {len(df) - len(df_clean):,} players")
print(f"\nCleaned group sizes:")
print(df_clean['version'].value_counts())
print(f"\nCleaned retention rates:")
retention_clean = df_clean.groupby('version')[['retention_1','retention_7']].mean().mul(100).round(2)
print(retention_clean)

Original dataset:  90,189 players
Cleaned dataset:   86,195 players
Players removed:   3,994 players

Cleaned group sizes:
version
gate_40    43432
gate_30    42763
Name: count, dtype: int64

Cleaned retention rates:
         retention_1  retention_7
version                          
gate_30        46.75        19.84
gate_40        46.22        19.03
